In [4]:
import sys
from pathlib import Path

In [5]:
src_path = Path("../src").resolve()
sys.path.append(str(src_path))

In [5]:
from api.events.models import EventModel
from api.db.session import engine
from sqlmodel import Session, select
from sqlalchemy import literal_column

ModuleNotFoundError: No module named 'api'

In [36]:
with Session(engine) as session:
    query = select(EventModel).order_by(EventModel.updated_at.asc()).limit(10)
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    print(compiled_query)
    print("")
    print(str(query))
    results = session.exec(query).fetchall()
    pprint(results)

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at ASC
 LIMIT 10

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at ASC
 LIMIT :param_1
[EventModel(description='', time=datetime.datetime(2025, 3, 21, 17, 0, 14, 793459, tzinfo=datetime.timezone.utc), id=1, updated_at=datetime.datetime(2025, 3, 21, 17, 0, 14, 793466, tzinfo=datetime.timezone.utc), page='pricing'),
 EventModel(description='', time=datetime.datetime(2025, 3, 21, 17, 0, 14, 808389, tzinfo=datetime.timezone.utc), id=2, updated_at=datetime.datetime(2025, 3, 21, 17, 0, 14, 808394, tzinfo=datetime.timezone.utc), page='/about'),
 EventModel(description='', time=datetime.datetime(2025, 3, 21, 17, 0, 14, 813534, tzinfo=datetime.timezone.utc), id=3, updated_at=datetime.datetime(2025, 3, 21, 17, 0, 14, 813538, tzinfo=datetime.timezone.ut

In [2]:
from sqlmodel import func 
from datetime import datetime, timedelta, timezone
from pprint import pprint


with Session(engine) as session:
    interval = literal_column("'1 minute'")
    bucket = func.time_bucket(interval, EventModel.created_at)
    pages = ['/about', '/contact', '/pages', '/pricing']
    start = datetime.now(timezone.utc) - timedelta(hours=1)
    finish = datetime.now(timezone.utc) + timedelta(hours=1)
    query = (
        select(
            bucket,
            EventModel.page,
            func.count()
        )
        .where(EventModel.created_at > start,
                EventModel.created_at <= finish,
                EventModel.page == "/about")
        .group_by(
            bucket,
            EventModel.page,
        )
        .order_by(
            bucket,
            EventModel.page,
        )
    )
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    # print(compiled_query)
    results = session.exec(query).fetchall()
    pprint(results)

NameError: name 'Session' is not defined